In [2]:
import pandas as pd, numpy as np
import lightgbm as lgb
from sklearn.model_selection import KFold
import lightgbm as lgb
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.decomposition import PCA
from catboost import CatBoostRegressor
import xgboost as xgb
# 1. Chargement des données (Utilise tes noms de fichiers sauvegardés)
X_train = pd.read_csv("../features/global_features/X_train.csv")
y_train = pd.read_csv("../features/global_features/y_train.csv")
X_test = pd.read_csv("../features/global_features/X_test.csv")

In [3]:
# 1. Identifier les colonnes CLIP (en supposant qu'elles contiennent 'clip' dans leur nom)
# Si elles n'ont pas de nom spécifique, adapte l'indexation (ex: X.iloc[:, :512])
clip_cols = [c for c in X_train.iloc[:,23:535].columns]
other_cols = [c for c in X_train.columns if c not in clip_cols]

print(f"Nombre de features CLIP détectées : {len(clip_cols)}")
print(f"Nombre d'autres features : {len(other_cols)}")
# 1. Standardiser CLIP avant la PCA
scaler = StandardScaler()
clip_scaled = X_train[clip_cols]#scaler.fit_transform(X_train[clip_cols])
clip_test_scaled = X_test[clip_cols]#scaler.transform(X_test[clip_cols])
# 2. Initialiser la PCA
# n_components=32 est un bon point de départ pour 512 dimensions sur 1600 vidéos
n_components = 15
pca = PCA(n_components=n_components, random_state=42)

# 3. Fit & Transform sur le TRAIN
clip_pca_train = pca.fit_transform(clip_scaled)

# 4. Transform uniquement sur le TEST (on n'utilise pas fit ici !)
clip_pca_test = pca.transform(clip_test_scaled)

# 5. Conversion en DataFrame pour reconstruction
clip_pca_train_df = pd.DataFrame(
    clip_pca_train, 
    columns=[f'pca_clip_{i}' for i in range(n_components)],
    index=X_train.index
)
clip_pca_test_df = pd.DataFrame(
    clip_pca_test, 
    columns=[f'pca_clip_{i}' for i in range(n_components)],
    index=X_test.index
)

# 6. Assemblage final : Autres features + Composantes PCA
X_train = pd.concat([X_train[other_cols], clip_pca_train_df], axis=1)
X_test = pd.concat([X_test[other_cols], clip_pca_test_df], axis=1)

print(f"Nouvelle forme de X_train : {X_train.shape}")
print(f"Variance expliquée cumulée : {pca.explained_variance_ratio_.sum():.2%}")

Nombre de features CLIP détectées : 512
Nombre d'autres features : 376
Nouvelle forme de X_train : (1348, 391)
Variance expliquée cumulée : 45.03%


In [4]:
# 1. Identifier les colonnes CLIP (en supposant qu'elles contiennent 'clip' dans leur nom)
# Si elles n'ont pas de nom spécifique, adapte l'indexation (ex: X.iloc[:, :512])
clip_cols = [c for c in X_train.iloc[:,-256:].columns]
other_cols = [c for c in X_train.columns if c not in clip_cols]

print(f"Nombre de features CLIP détectées : {len(clip_cols)}")
print(f"Nombre d'autres features : {len(other_cols)}")
# 1. Standardiser CLIP avant la PCA
scaler = StandardScaler()
clip_scaled = X_train[clip_cols]#scaler.fit_transform(X_train[clip_cols])
clip_test_scaled = X_test[clip_cols]#scaler.transform(X_test[clip_cols])
# 2. Initialiser la PCA
# n_components=32 est un bon point de départ pour 512 dimensions sur 1600 vidéos
n_components = 15
pca = PCA(n_components=n_components, random_state=42)

# 3. Fit & Transform sur le TRAIN
clip_pca_train = pca.fit_transform(clip_scaled)

# 4. Transform uniquement sur le TEST (on n'utilise pas fit ici !)
clip_pca_test = pca.transform(clip_test_scaled)

# 5. Conversion en DataFrame pour reconstruction
clip_pca_train_df = pd.DataFrame(
    clip_pca_train, 
    columns=[f'pca_clip_{i}' for i in range(n_components)],
    index=X_train.index
)
clip_pca_test_df = pd.DataFrame(
    clip_pca_test, 
    columns=[f'pca_clip_{i}' for i in range(n_components)],
    index=X_test.index
)

# 6. Assemblage final : Autres features + Composantes PCA
X_train = pd.concat([X_train[other_cols], clip_pca_train_df], axis=1)
X_test = pd.concat([X_test[other_cols], clip_pca_test_df], axis=1)

print(f"Nouvelle forme de X_train : {X_train.shape}")
print(f"Variance expliquée cumulée : {pca.explained_variance_ratio_.sum():.2%}")

Nombre de features CLIP détectées : 256
Nombre d'autres features : 135
Nouvelle forme de X_train : (1348, 150)
Variance expliquée cumulée : 79.69%


In [5]:
# 2. Préparation
y_train = y_train.values.flatten()
test_ids = X_test['video_id']
X = X_train.drop(columns=['video_id'], errors='ignore')
X_test_final = X_test.drop(columns=['video_id'], errors='ignore')

# 3. Configuration du K-Fold
n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

# Listes pour stocker les scores de validation et les prédictions finales
oof_preds = np.zeros(len(X)) # Out-of-fold predictions
test_preds = np.zeros(len(X_test_final))
cv_scores = []

In [ ]:
X_train = pd.DataFrame(X_train)
y_train = pd.DataFrame(y_train)
X_test = pd.DataFrame(X_test)

In [7]:
# On isole l'ID pour la soumission finale (très important !)
test_ids = X_test['video_id'].copy()

# On définit les features en supprimant video_id
# errors='ignore' permet de ne pas planter si la colonne est déjà absente
X_train = X_train.drop(columns=['video_id'], errors='ignore')
X_test = X_test.drop(columns=['video_id'], errors='ignore')

# On s'assure que y_train est un array 1D pour les calculs de metrics
# y_train doit être la colonne 'score' uniquement
y_train_values = y_train.values.flatten()

In [8]:
# =========================
# 3) LightGBM params
# =========================
lgb_params = {
    "objective": "regression",
    "metric": "rmse",
    "learning_rate": 0.015,     # Plus lent pour ne pas rater l'optimum
    "num_leaves": 15,          # Très bas pour éviter l'overfitting
    "max_depth": 4,            # On force des arbres courts (plus robustes)
    "min_child_samples": 40,   # On force chaque feuille à avoir au moins 40 vidéos
    "feature_fraction": 0.5,   # On ne prend que 50% des colonnes par arbre
    "reg_alpha": 1.0,          # L1 plus fort
    "reg_lambda": 5.0,         # L2 plus fort
    "verbose": -1,
    "random_state": 42
}
# =========================
# 4) Cross-validation + OOF
# =========================
kf = KFold(n_splits=5, shuffle=True, random_state=42)

oof_preds   = np.zeros(len(X_train))
test_preds  = np.zeros(len(X_test))
feature_cols = X_train.columns # Maintenant sans video_id
feature_imp = np.zeros(len(feature_cols))

print(f"\n{'='*50}")
print(f"KFold CV — 5 folds")
print(f"{'='*50}")

for fold, (train_idx, val_idx) in enumerate(kf.split(X_train, y_train_values)):
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train_values[train_idx], y_train_values[val_idx]

    model = lgb.LGBMRegressor(n_estimators=3000, **lgb_params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[
            lgb.early_stopping(stopping_rounds=100, verbose=False),
            lgb.log_evaluation(period=200)
        ]
    )

    oof_preds[val_idx] = model.predict(X_val)
    test_preds         += model.predict(X_test) / 5
    feature_imp        += model.feature_importances_ / 5

    fold_rmse = np.sqrt(mean_squared_error(y_val, oof_preds[val_idx]))
    print(f"  Fold {fold+1} | Best iter: {model.best_iteration_:4d} | RMSE: {fold_rmse:.4f}")

# Utilisation de y_train_values pour le calcul final
oof_rmse = np.sqrt(mean_squared_error(y_train_values, oof_preds))
print(f"\n{'='*50}")
print(f"OOF RMSE global : {oof_rmse:.4f}")
print(f"{'='*50}")


KFold CV — 5 folds
[200]	valid_0's rmse: 1.18674
[400]	valid_0's rmse: 1.15512
[600]	valid_0's rmse: 1.1513
  Fold 1 | Best iter:  548 | RMSE: 1.1501
[200]	valid_0's rmse: 1.33339
[400]	valid_0's rmse: 1.28245
[600]	valid_0's rmse: 1.27279
[800]	valid_0's rmse: 1.26806
[1000]	valid_0's rmse: 1.26743
  Fold 2 | Best iter:  980 | RMSE: 1.2668
[200]	valid_0's rmse: 1.24228
[400]	valid_0's rmse: 1.22152
  Fold 3 | Best iter:  433 | RMSE: 1.2195
[200]	valid_0's rmse: 1.28205
[400]	valid_0's rmse: 1.23723
[600]	valid_0's rmse: 1.2311
[800]	valid_0's rmse: 1.2301
[1000]	valid_0's rmse: 1.22825
  Fold 4 | Best iter:  977 | RMSE: 1.2272
[200]	valid_0's rmse: 1.34589
[400]	valid_0's rmse: 1.29923
  Fold 5 | Best iter:  407 | RMSE: 1.2988

OOF RMSE global : 1.2335


In [9]:
# =========================
# 5) Feature importance top 20
# =========================
fi_df = pd.DataFrame({"feature": feature_cols, "importance": feature_imp})
fi_df = fi_df.sort_values("importance", ascending=False).head(20)
print("\nTop 20 features:")
print(fi_df.to_string(index=False))


Top 20 features:
                             feature  importance
                          pca_clip_4       205.4
                          pca_clip_3       200.4
                          pca_clip_2       199.0
             up_uploader_val_thorens       147.0
                          pca_clip_1       144.6
                          pca_clip_0       129.2
                        release_year       114.6
                         pca_clip_10        85.0
                          vggish_001        82.8
                            text_len        82.8
                          pca_clip_8        78.6
                                  W2        69.6
                         mfcc_9_mean        67.2
                 up_uploader_avoriaz        65.6
up_uploader_stanton_arlberg_official        61.4
                        att_skewness        61.0
                       f1_brightness        60.4
                         mfcc_7_mean        59.6
                             rms_std        58.8
  

In [10]:
# 1. Paramètres optimisés pour ton petit dataset (1348 lignes)
cb_params = {
    'iterations': 2000,          # Nombre maximum d'arbres
    'learning_rate': 0.02,       # Pas d'apprentissage lent pour la précision
    'depth': 6,                  # Profondeur modérée pour éviter d'apprendre le bruit
    'l2_leaf_reg': 5,            # Régularisation L2 forte
    'loss_function': 'RMSE',
    'eval_metric': 'RMSE',
    'random_seed': 42,
    'verbose': 200,              # Affiche le score tous les 200 arbres
    'early_stopping_rounds': 100 # Stop si le RMSE ne baisse plus pendant 100 tours
}

# 2. Préparation des tableaux de résultats
cb_oof_preds = np.zeros(len(X_train))
cb_test_preds = np.zeros(len(X_test))

print(f"\n{'='*50}")
print(f"CatBoost KFold CV — 5 folds")
print(f"{'='*50}")

# 3. Boucle K-Fold
for fold, (train_idx, val_idx) in enumerate(kf.split(X_train, y_train_values)):
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train_values[train_idx], y_train_values[val_idx]

    model_cb = CatBoostRegressor(**cb_params)
    
    model_cb.fit(
        X_tr, y_tr,
        eval_set=(X_val, y_val),
        use_best_model=True,
        plot=False # Met à True si tu es sur Jupyter pour voir la courbe en temps réel
    )

    cb_oof_preds[val_idx] = model_cb.predict(X_val)
    cb_test_preds         += model_cb.predict(X_test) / 5

    fold_rmse = np.sqrt(mean_squared_error(y_val, cb_oof_preds[val_idx]))
    print(f"  Fold {fold+1} | RMSE: {fold_rmse:.4f}")

cb_oof_rmse = np.sqrt(mean_squared_error(y_train_values, cb_oof_preds))
print(f"\nOOF CatBoost RMSE global : {cb_oof_rmse:.4f}")


CatBoost KFold CV — 5 folds
0:	learn: 1.7020456	test: 1.5748585	best: 1.5748585 (0)	total: 75.3ms	remaining: 2m 30s
200:	learn: 1.0957199	test: 1.2140209	best: 1.2140209 (200)	total: 2.65s	remaining: 23.7s
400:	learn: 0.8880675	test: 1.1685901	best: 1.1685901 (400)	total: 4.96s	remaining: 19.8s
600:	learn: 0.7173096	test: 1.1569533	best: 1.1566989 (580)	total: 7.57s	remaining: 17.6s
800:	learn: 0.5835053	test: 1.1537985	best: 1.1531266 (757)	total: 9.81s	remaining: 14.7s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 1.153126588
bestIteration = 757

Shrink model to first 758 iterations.
  Fold 1 | RMSE: 1.1531
0:	learn: 1.6638228	test: 1.7310956	best: 1.7310956 (0)	total: 28.3ms	remaining: 56.5s
200:	learn: 1.0746161	test: 1.3697954	best: 1.3696303 (198)	total: 3.8s	remaining: 34s
400:	learn: 0.8786864	test: 1.3223470	best: 1.3223470 (400)	total: 7.55s	remaining: 30.1s
600:	learn: 0.7055774	test: 1.3027017	best: 1.3022224 (584)	total: 11.5s	remaining: 26.8s
800:	le

In [11]:
final_preds = (0.5 * cb_test_preds) + \
                       (0.5 * test_preds)

# 5. Sauvegarde de la nouvelle soumission
submission = pd.DataFrame({
    'ID': test_ids,
    'popularity': final_preds
})
submission.to_csv("submission_cat_lgbm3.csv", index=False)